# Step 3.2.1: 数据加载与特征工程 (Data Loading & Feature Engineering)

## 📋 数据准备说明

**重要**: 本notebook使用Gradient Sports数据集中的Rosters数据作为球员位置信息来源。

### 必需的数据文件

本notebook需要以下数据文件：

1. **Rosters数据** - 每场比赛的球员阵容和位置信息
   - 位置: `E:\JerryWu\Master\SoccerAnalytics\OpenData\TrackingData\Gradient Sports  Enhanced 2022 World Cup Dataset\Rosters\`
   - 文件格式: JSON文件，文件名为比赛ID（如10517.json）
   - 包含信息:
     - 球员ID (player.id)
     - 球员姓名 (player.nickname)
     - 详细位置 (positionGroupType): GK, LCB, RCB, LWB, RWB, MCB, LB, RB, DM, CM, AM, LW, RW, CF等
     - 球衣号码 (shirtNumber)
     - 是否首发 (started)
     - 所属球队 (team.id, team.name)

2. **博彩赔率数据** - 反映赛前实力对比
   - 位置: `E:\JerryWu\Master\SoccerAnalytics\OpenData\2022 WC prediction\Odds\`
   - 文件: `2022WC odds group stages.xlsx` 和 `2022WC odds knockout stages.xlsx`
   - 包含: 主胜赔率、平局赔率、客胜赔率

---

## 目标

本notebook为B-GNN模型准备训练数据，整合以下数据源：

1. **Shape Graph序列** (来自2.3)
2. **战术阶段上下文** (来自1.3)
3. **实时比赛情境** (来自gradient sports数据集)
4. **球员位置数据** (来自Rosters数据)
5. **FIFA排名** (球队实力)
6. **博彩赔率** (赛前实力对比)
7. **EFPI基准结果** (作为伪标签)

## 核心创新

- **多源特征融合**: 将球员个体特征、图结构特征、战术上下文特征整合
- **情境感知**: 利用战术阶段、比分、时间等情境信息
- **球员位置编码**: 使用Rosters数据中的详细位置信息作为节点特征
- **完整阵型库**: 使用EFPI的全部阵型模板作为输出类别

## 1. 导入库和配置

In [1]:
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 数据处理
import numpy as np
import pandas as pd
import polars as pl
import pickle
from tqdm import tqdm
import json

# 图神经网络
import torch
import torch_geometric
from torch_geometric.data import Data, Dataset
import networkx as nx

# 可视化
import matplotlib.pyplot as plt
import seaborn as sns

# 配置
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', None)

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

print(f"✅ 库导入成功")
print(f"PyTorch版本: {torch.__version__}")
print(f"PyTorch Geometric版本: {torch_geometric.__version__}")

✅ 库导入成功
PyTorch版本: 2.9.1+cpu
PyTorch Geometric版本: 2.7.0


## 2. 配置路径和参数

In [2]:
# 比赛信息
GAME_ID = "10517"  # 2022世界杯决赛
HOME_TEAM_ID = '364'  # 阿根廷
AWAY_TEAM_ID = '363'  # 法国

# 数据路径
DATA_DIR = Path('../../../data/morph_test')
GRAPHS_DIR = DATA_DIR / 'shape_graphs' / 'graphs'
OUTPUT_DIR = DATA_DIR / 'bgnn_data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 外部数据路径
EXTERNAL_DATA_DIR = Path('E:/JerryWu/Master/SoccerAnalytics/OpenData')
GRADIENT_SPORTS_DIR = EXTERNAL_DATA_DIR / 'TrackingData' / 'Gradient Sports  Enhanced 2022 World Cup Dataset'

# Rosters数据路径
ROSTERS_DIR = GRADIENT_SPORTS_DIR / 'Rosters'
ROSTER_FILE = ROSTERS_DIR / f'{GAME_ID}.json'

# FIFA排名和博彩赔率
FIFA_RANKING_FILE = EXTERNAL_DATA_DIR / '2022 WC prediction' / 'FIFA ranking' / 'FIFA ranking details 2022-10-06.xlsx'
ODDS_GROUP_FILE = EXTERNAL_DATA_DIR / '2022 WC prediction' / 'Odds' / '2022WC odds group stages.xlsx'
ODDS_KNOCKOUT_FILE = EXTERNAL_DATA_DIR / '2022 WC prediction' / 'Odds' / '2022WC odds knockout stages.xlsx'

print(f"✅ 配置完成")
print(f"比赛: 2022世界杯决赛 (Game {GAME_ID})")
print(f"分析球队: 阿根廷 (Team ID: {HOME_TEAM_ID})")
print(f"输出目录: {OUTPUT_DIR}")
print(f"Rosters文件: {ROSTER_FILE}")

✅ 配置完成
比赛: 2022世界杯决赛 (Game 10517)
分析球队: 阿根廷 (Team ID: 364)
输出目录: ..\..\..\data\morph_test\bgnn_data
Rosters文件: E:\JerryWu\Master\SoccerAnalytics\OpenData\TrackingData\Gradient Sports  Enhanced 2022 World Cup Dataset\Rosters\10517.json


## 3. 加载EFPI阵型库

从EFPI结果中自动提取完整的阵型模板作为B-GNN的输出类别

In [3]:
print("=== 加载完整的 EFPI 阵型库（65种） ===")

# 从 mplsoccer.Pitch 对象加载完整的 65 种阵型模板
from mplsoccer import Pitch

# 创建一个Pitch对象来访问formations
pitch = Pitch(pitch_type='secondspectrum', pitch_length=105, pitch_width=68)

# mplsoccer.Pitch.formations 是一个列表，包含 EFPI 论文中的 65 种阵型
# 过滤掉纯字母阵型名（如'flat'），只保留数字阵型
all_formations = sorted([f for f in pitch.formations if not f.isalpha()])

print(f"✅ 从 mplsoccer.Pitch 加载了 {len(all_formations)} 种阵型模板")
print(f"   这是 EFPI 论文 Appendix A 中的完整阵型库")

# 加载EFPI结果以分析实际出现的阵型
efpi_file = DATA_DIR / 'efpi_baseline' / f'efpi_formation_sequence_{GAME_ID}_fullmatch.parquet'
if efpi_file.exists():
    efpi_results = pd.read_parquet(efpi_file)
    observed_formations = set(efpi_results['formation'].unique())
    
    print(f"\n📊 阵型覆盖分析:")
    print(f"  - 完整阵型库（EFPI论文）: {len(all_formations)} 种")
    print(f"  - 决赛中实际出现: {len(observed_formations)} 种")
    
    # 检查是否有观察到的阵型不在完整库中
    extra_formations = observed_formations - set(all_formations)
    if len(extra_formations) > 0:
        print(f"  ⚠️ 警告: {len(extra_formations)} 种阵型在完整库中未定义:")
        for formation in sorted(extra_formations):
            print(f"      - {formation}")
        # 将这些阵型添加到完整库中
        all_formations = sorted(list(set(all_formations) | extra_formations))
        print(f"  ✅ 已将这些阵型添加到完整库，新总数: {len(all_formations)} 种")
    
    # 找出未在决赛中出现的阵型
    missing_formations = set(all_formations) - observed_formations
    print(f"  - 决赛中未出现: {len(missing_formations)} 种")
    
    if len(missing_formations) > 0 and len(missing_formations) <= 20:
        print(f"\n未在决赛中出现的阵型:")
        for formation in sorted(missing_formations):
            print(f"    - {formation}")
    
    # 显示阵型分布（前15）
    print(f"\n决赛中阵型分布（前15）:")
    print(efpi_results['formation'].value_counts().head(15))
    
else:
    print(f"⚠️ EFPI文件不存在: {efpi_file}")
    print("请先运行3.1.1_test_EFPI_Formation_Identification.ipynb")
    efpi_results = None
    observed_formations = set()

# 创建完整的阵型映射
formation_to_idx = {formation: idx for idx, formation in enumerate(all_formations)}
idx_to_formation = {idx: formation for formation, idx in formation_to_idx.items()}

print(f"\n完整阵型库（前30种）:")
print(all_formations[:30])

# 保存完整的阵型映射
formation_mapping = {
    'formations': all_formations,
    'formation_to_idx': formation_to_idx,
    'idx_to_formation': idx_to_formation,
    'n_formations': len(all_formations),
    'source': 'mplsoccer_pitch',
    'observed_formations': sorted(list(observed_formations)),
    'n_observed': len(observed_formations),
    'missing_formations': sorted(list(set(all_formations) - observed_formations)),
    'n_missing': len(set(all_formations) - observed_formations)
}

with open(OUTPUT_DIR / 'formation_mapping.pkl', 'wb') as f:
    pickle.dump(formation_mapping, f)
print(f"\n✅ 完整阵型映射已保存: {OUTPUT_DIR / 'formation_mapping.pkl'}")
print(f"   包含 {len(all_formations)} 种阵型类别（EFPI 完整库）")

print("\n⚠️ 重要说明:")
print("  1. B-GNN 模型将使用完整的阵型库作为输出类别")
print(f"  2. 训练时，{len(set(all_formations) - observed_formations)} 种阵型在训练集中样本数为 0")
print("  3. 这是类别不平衡问题，需要在训练时考虑:")
print("     - 使用 class_weight 为少数类赋予更高权重")
print("     - 使用 Focal Loss 关注难分类样本")
print("     - 评估时使用 macro-F1 而非 accuracy")
print("  4. 这样可以确保模型与 EFPI 基准的输出空间一致（公平对比）")

=== 加载完整的 EFPI 阵型库（65种） ===
✅ 从 mplsoccer.Pitch 加载了 65 种阵型模板
   这是 EFPI 论文 Appendix A 中的完整阵型库

📊 阵型覆盖分析:
  - 完整阵型库（EFPI论文）: 65 种
  - 决赛中实际出现: 50 种
  - 决赛中未出现: 15 种

未在决赛中出现的阵型:
    - 341
    - 3411
    - 342
    - 351
    - 41221
    - 4131
    - 4212
    - 422
    - 4221
    - 431
    - 4311
    - 432
    - 44
    - 53
    - 531

决赛中阵型分布（前15）:
formation
442        13460
433         8104
1234        5975
343         5855
42211       5264
4321        5145
42121       5116
2431        4833
4141        4404
4132        4106
343flat     4090
3511        4007
4222        3634
31213       3598
541         3564
Name: count, dtype: int64

完整阵型库（前30种）:
['1234', '1324', '1423', '1432', '2233', '2332', '2413', '2422', '2431', '312112', '31213', '31222', '31312', '3142', '32122', '32221', '3232', '3241', '3322', '3331', '341', '3411', '3412', '3412flat', '342', '3421', '3421flat', '343', '343flat', '351']

✅ 完整阵型映射已保存: ..\..\..\data\morph_test\bgnn_data\formation_mapping.pkl
   包含 65 种阵型类别（EFPI 完整

## 4. 加载基础数据

### 4.1 加载追踪数据和战术标签

In [4]:
print("\n=== 加载追踪数据和战术标签 ===")

# 加载带战术标签的追踪数据
tracking_file = DATA_DIR / f"tracking_data_{GAME_ID}_tactical_labels.parquet"
if tracking_file.exists():
    tracking_data = pl.read_parquet(tracking_file)
    print(f"✅ 追踪数据: {tracking_data.shape}")
    
    # 检查战术标签列
    if 'macro_phase' in tracking_data.columns and 'fine_intent' in tracking_data.columns:
        print("✅ 检测到战术标签（macro_phase和fine_intent）")
    else:
        print("⚠️ 未检测到战术标签")
        
    display(tracking_data.head())
else:
    print(f"❌ 文件不存在: {tracking_file}")


=== 加载追踪数据和战术标签 ===
✅ 追踪数据: (2966182, 25)
✅ 检测到战术标签（macro_phase和fine_intent）


period_id,timestamp,frame_id,ball_state,id,x,y,z,team_id,position_name,game_id,vx,vy,vz,v,ax,ay,az,a,ball_owning_team_id,is_ball_carrier,attack_intent,defense_intent,macro_phase,fine_intent
i64,duration[μs],i64,str,str,f64,f64,f64,str,str,i32,f64,f64,f64,f64,f64,f64,f64,f64,str,bool,str,str,str,str
1,821µs,4630,"""alive""","""10715""",4.987,-1.993,0.0,"""364""","""ST""",10517,-0.522823,0.079386,0.0,0.528815,0.0,0.0,0.0,0.0,"""363""",false,"""ATTACKING_PLAY""","""MID_BLOCK""","""Out-of-Possession Open Play""","""MID_BLOCK"""
1,821µs,4630,"""alive""","""11""",-18.172,3.632,0.0,"""364""","""LCB""",10517,-1.114241,-0.985899,0.0,1.487794,0.0,0.0,0.0,0.0,"""363""",false,"""ATTACKING_PLAY""","""MID_BLOCK""","""Out-of-Possession Open Play""","""MID_BLOCK"""
1,821µs,4630,"""alive""","""113""",0.763,-18.099,0.0,"""363""","""ST""",10517,-2.287401,-0.437484,0.0,2.328862,0.0,0.0,0.0,0.0,"""363""",false,"""ATTACKING_PLAY""","""MID_BLOCK""","""In-Possession Open Play""","""ATTACKING_PLAY"""
1,821µs,4630,"""alive""","""11856""",-6.252,3.173,0.0,"""364""","""CM""",10517,-1.145117,-1.007385,0.0,1.525161,0.0,0.0,0.0,0.0,"""363""",false,"""ATTACKING_PLAY""","""MID_BLOCK""","""Out-of-Possession Open Play""","""MID_BLOCK"""
1,821µs,4630,"""alive""","""13222""",3.636,18.712,0.0,"""364""","""RB""",10517,-0.874427,-1.040521,0.0,1.359156,0.0,0.0,0.0,0.0,"""363""",false,"""ATTACKING_PLAY""","""MID_BLOCK""","""Out-of-Possession Open Play""","""MID_BLOCK"""


### 4.2 加载Shape Graph数据

In [5]:
print("\n=== 加载Shape Graph数据 ===")

# 加载Shape Graph文件列表
graph_files = sorted(GRAPHS_DIR.glob('shape_graph_*.pkl'))
print(f"找到 {len(graph_files)} 个Shape Graph文件")

if len(graph_files) > 0:
    # 加载第一个文件查看结构
    with open(graph_files[0], 'rb') as f:
        sample_graph = pickle.load(f)
    
    print(f"\nShape Graph数据结构:")
    print(f"  - frame_id: {sample_graph['frame_id']}")
    print(f"  - positions shape: {sample_graph['positions'].shape}")
    print(f"  - graph nodes: {sample_graph['graph'].number_of_nodes()}")
    print(f"  - graph edges: {sample_graph['graph'].number_of_edges()}")
else:
    print("❌ 未找到Shape Graph文件")


=== 加载Shape Graph数据 ===
找到 50084 个Shape Graph文件

Shape Graph数据结构:
  - frame_id: 10018
  - positions shape: (10, 2)
  - graph nodes: 10
  - graph edges: 15


### 4.2.1 加载Shape Graph位置识别结果

从3.1.2.2的批处理结果中加载球员位置信息（垂直层级和水平位置）

In [6]:
print("\n=== 加载Shape Graph位置识别结果 ===")

# 加载3.1.2.2的批处理结果
shapegraphs_file = DATA_DIR / 'shapegraphs_baseline' / f'shapegraphs_baseline_results_{GAME_ID}.parquet'
if shapegraphs_file.exists():
    shapegraphs_results = pd.read_parquet(shapegraphs_file)
    print(f"✅ Shape Graph位置识别结果: {shapegraphs_results.shape}")
    
    # 显示数据结构
    print(f"\n列名: {shapegraphs_results.columns.tolist()}")
    print(f"\n前5行:")
    display(shapegraphs_results.head())
    
    # 统计垂直层级分布
    print(f"\n垂直层级分布示例（第一帧）:")
    first_frame_levels = shapegraphs_results.iloc[0]['vertical_levels']
    print(f"  {first_frame_levels}")
    
    # 统计水平位置分布
    print(f"\n水平位置分布示例（第一帧）:")
    first_frame_positions = shapegraphs_results.iloc[0]['horizontal_positions']
    print(f"  {first_frame_positions}")
    
else:
    print(f"⚠️ Shape Graph位置识别结果文件不存在: {shapegraphs_file}")
    print("请先运行3.1.2.2_test_ShapeGraphs_Batch_Processing.ipynb")
    shapegraphs_results = None


=== 加载Shape Graph位置识别结果 ===
✅ Shape Graph位置识别结果: (50084, 14)

列名: ['frame_id', 'formation', 'formation_code', 'formation_detailed', 'n_defenders', 'n_dm', 'n_midfielders', 'n_am', 'n_forwards', 'vertical_levels', 'horizontal_positions', 'formation_smoothed', 'formation_code_smoothed', 'formation_detailed_smoothed']

前5行:


,frame_id,formation,formation_code,formation_detailed,n_defenders,n_dm,n_midfielders,n_am,n_forwards,vertical_levels,horizontal_positions,formation_smoothed,formation_code_smoothed,formation_detailed_smoothed
0,10018,2-5-3,2(014)3,20143,1,1,1,4,3,"[F, DM, AM, M, F, B, AM, AM, F, AM]","[R, C, L, C, L, C, RC, LC, C, R]",2-5-3,2(014)3,20143
1,10019,2-5-3,2(014)3,20143,1,1,1,4,3,"[F, DM, AM, M, F, B, AM, AM, F, AM]","[R, C, L, C, L, C, RC, LC, C, R]",2-5-3,2(014)3,20143
2,10020,2-5-3,2(014)3,20143,1,1,1,4,3,"[F, DM, AM, M, F, B, AM, AM, F, AM]","[R, C, L, C, L, C, RC, LC, C, R]",2-5-3,2(014)3,20143
3,10021,2-5-3,2(014)3,20143,1,1,1,4,3,"[F, DM, AM, M, F, B, AM, AM, F, AM]","[R, C, L, C, L, C, RC, LC, C, R]",2-5-3,2(014)3,20143
4,10022,3-5-2,3(212)2,32122,3,2,1,2,2,"[F, B, AM, B, F, B, DM, DM, AM, M]","[R, C, R, L, L, R, R, L, L, C]",3-5-2,3(212)2,32122



垂直层级分布示例（第一帧）:
  ['F' 'DM' 'AM' 'M' 'F' 'B' 'AM' 'AM' 'F' 'AM']

水平位置分布示例（第一帧）:
  ['R' 'C' 'L' 'C' 'L' 'C' 'RC' 'LC' 'C' 'R']


### 4.3 加载Rosters数据（球员位置信息）

In [7]:
print("\n=== 加载Rosters数据 ===")

if ROSTER_FILE.exists():
    with open(ROSTER_FILE, 'r', encoding='utf-8') as f:
        roster_data = json.load(f)
    
    print(f"✅ Rosters数据: {len(roster_data)} 名球员（双方球队）")
    
    # 转换为DataFrame便于查询
    roster_df = pd.DataFrame(roster_data)
    
    # 展开player和team嵌套字段
    roster_df['player_id'] = roster_df['player'].apply(lambda x: x['id'])
    roster_df['player_nickname'] = roster_df['player'].apply(lambda x: x['nickname'])
    roster_df['team_id'] = roster_df['team'].apply(lambda x: x['id'])
    roster_df['team_name'] = roster_df['team'].apply(lambda x: x['name'])
    
    # 显示关键列
    key_cols = ['player_id', 'player_nickname', 'positionGroupType', 'shirtNumber', 
                'started', 'team_id', 'team_name']
    print(f"\n关键列: {key_cols}")
    display(roster_df[key_cols].head(10))
    
    # 统计位置分布
    print(f"\n位置分布:")
    position_counts = roster_df['positionGroupType'].value_counts()
    for pos, count in position_counts.items():
        print(f"   - {pos}: {count}")
    
    # 按球队分组统计
    print(f"\n按球队统计:")
    for team_id in roster_df['team_id'].unique():
        team_players = roster_df[roster_df['team_id'] == team_id]
        team_name = team_players.iloc[0]['team_name']
        print(f"   - {team_name} (ID: {team_id}): {len(team_players)} 名球员")
        print(f"     首发: {len(team_players[team_players['started']==True])} 名")
        print(f"     替补: {len(team_players[team_players['started']==False])} 名")
    
else:
    print(f"⚠️ Rosters数据文件不存在: {ROSTER_FILE}")
    roster_df = None


=== 加载Rosters数据 ===
✅ Rosters数据: 50 名球员（双方球队）

关键列: ['player_id', 'player_nickname', 'positionGroupType', 'shirtNumber', 'started', 'team_id', 'team_name']


,player_id,player_nickname,positionGroupType,shirtNumber,started,team_id,team_name
0,13222,Nahuel Molina,RB,26,True,364,Argentina
1,4622,Kingsley Coman,RW,20,False,363,France
2,1675,Raphael Varane,RCB,4,True,363,France
3,8058,Paulo Dybala,CF,21,False,364,Argentina
4,8025,Nicolas Tagliafico,LB,3,True,364,Argentina
5,1965,Alphonse Areola,GK,23,False,363,France
6,1389,Angel Correa,CF,15,False,364,Argentina
7,10715,Julian Alvarez,CF,9,True,364,Argentina
8,10756,Jordan Veretout,CM,15,False,363,France
9,3888,Aurélien Tchouaméni,CM,8,True,363,France



位置分布:
   - CM: 10
   - CF: 6
   - RCB: 6
   - GK: 6
   - RB: 5
   - RW: 3
   - LB: 3
   - LCB: 3
   - AM: 3
   - LW: 3
   - DM: 2

按球队统计:
   - Argentina (ID: 364): 26 名球员
     首发: 11 名
     替补: 15 名
   - France (ID: 363): 24 名球员
     首发: 11 名
     替补: 13 名


### 4.4 加载FIFA排名

In [8]:
print("\n=== 加载FIFA排名 ===")

if FIFA_RANKING_FILE.exists():
    fifa_df = pd.read_excel(FIFA_RANKING_FILE)
    print(f"✅ FIFA排名数据: {fifa_df.shape}")
    print(f"\n列名: {fifa_df.columns.tolist()}")
    
    # 查找阿根廷和法国的排名
    if 'country_full' in fifa_df.columns or 'country' in fifa_df.columns:
        country_col = 'country_full' if 'country_full' in fifa_df.columns else 'country'
        arg_ranking = fifa_df[fifa_df[country_col].str.contains('Argentina', case=False, na=False)]
        fra_ranking = fifa_df[fifa_df[country_col].str.contains('France', case=False, na=False)]
        
        print(f"\n阿根廷排名:")
        display(arg_ranking)
        print(f"\n法国排名:")
        display(fra_ranking)
else:
    print(f"⚠️ FIFA排名文件不存在: {FIFA_RANKING_FILE}")
    fifa_df = None


=== 加载FIFA排名 ===


✅ FIFA排名数据: (211, 8)

列名: ['rank', 'country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date']

阿根廷排名:


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
2,3,Argentina,ARG,1774,1771,0,CONMEBOL,2022-10-06



法国排名:


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
3,4,France,FRA,1760,1765,0,UEFA,2022-10-06


### 4.5 加载博彩赔率

In [9]:
print("\n=== 加载博彩赔率 ===")

# 决赛属于淘汰赛阶段
if ODDS_KNOCKOUT_FILE.exists():
    odds_df = pd.read_excel(ODDS_KNOCKOUT_FILE)
    print(f"✅ 博彩赔率数据: {odds_df.shape}")
    print(f"\n列名: {odds_df.columns.tolist()}")
    
    # 查找决赛赔率
    if 'match' in odds_df.columns or 'Match' in odds_df.columns:
        match_col = 'match' if 'match' in odds_df.columns else 'Match'
        final_odds = odds_df[odds_df[match_col].str.contains('Argentina.*France|France.*Argentina', case=False, na=False)]
        
        if len(final_odds) > 0:
            print(f"\n决赛赔率:")
            display(final_odds)
        else:
            print("\n⚠️ 未找到决赛赔率数据")
else:
    print(f"⚠️ 博彩赔率文件不存在: {ODDS_KNOCKOUT_FILE}")
    odds_df = None


=== 加载博彩赔率 ===
✅ 博彩赔率数据: (16, 6)

列名: ['Round', 'Home', 'Away', 'Home win', 'Draw', 'Away win']


## 5. 球员ID映射

使用Rosters数据中的球员ID直接与追踪数据匹配

In [10]:
print("=== 创建球员ID到位置的映射 ===")

if roster_df is not None:
    # 创建球员ID到位置的映射字典
    player_position_map = {}
    
    for _, row in roster_df.iterrows():
        player_id = row['player_id']
        player_position_map[player_id] = {
            'positionGroupType': row['positionGroupType'],
            'player_nickname': row['player_nickname'],
            'shirtNumber': row['shirtNumber'],
            'started': row['started'],
            'team_id': row['team_id'],
            'team_name': row['team_name']
        }
    
    print(f"✅ 创建了 {len(player_position_map)} 名球员的位置映射")
    
    # 显示阿根廷球员的位置信息
    arg_players = roster_df[roster_df['team_id'] == HOME_TEAM_ID]
    print(f"\n阿根廷球员位置信息:")
    display(arg_players[['player_nickname', 'positionGroupType', 'shirtNumber', 'started']].sort_values('shirtNumber'))
    
else:
    print("⚠️ 跳过球员映射：Rosters数据未加载")
    player_position_map = {}

=== 创建球员ID到位置的映射 ===
✅ 创建了 50 名球员的位置映射

阿根廷球员位置信息:


,player_nickname,positionGroupType,shirtNumber,started
33,Franco Armani,GK,1,False
46,Lionel Messi,RW,10,True
42,Ángel Di María,LW,11,True
14,Geronimo Rulli,GK,12,False
15,Cristian Romero,RCB,13,True
32,Exequiel Palacios,CM,14,False
6,Angel Correa,CF,15,False
25,Thiago Almada,AM,16,False
22,Papu Gómez,AM,17,False
20,Guido Rodríguez,DM,18,False


## 6. 特征工程准备

为每一帧准备特征向量

In [11]:
print("=== 特征工程准备 ===")

# 定义Rosters位置体系（基于实际数据，共16种）
ALL_ROSTER_POSITIONS = [
    'GK',                                    # 守门员
    'LB', 'LCB', 'MCB', 'RCB', 'RB',       # 后卫
    'LWB', 'RWB',                           # 翼卫
    'LM', 'DM', 'CM', 'AM', 'RM',          # 中场
    'LW', 'CF', 'RW',                       # 前锋
]

print(f"\nRosters位置体系（共{len(ALL_ROSTER_POSITIONS)}个位置）:")
print(f"  守门员(1): GK")
print(f"  后卫(5): LB, LCB, MCB, RCB, RB")
print(f"  翼卫(2): LWB, RWB")
print(f"  中场(5): LM, DM, CM, AM, RM")
print(f"  前锋(3): LW, CF, RW")

# 定义5×5位置矩阵映射
# 矩阵维度: [纵向层级(5), 横向位置(5)]
# 纵向层级（从后到前）:
#   0: GK (守门员)
#   1: D (后卫线)
#   2: DM (防守中场)
#   3: M/AM (中场/攻击中场)
#   4: F (前锋线)
# 横向位置（从左到右）:
#   0: L (左)
#   1: LC (左中)
#   2: C (中)
#   3: RC (右中)
#   4: R (右)

POSITION_MATRIX_MAPPING = {
    # 守门员 - 第0层，中央
    'GK': [(0, 2, 1.0)],
    
    # 后卫 - 第1层
    'LB': [(1, 0, 1.0)],      # 左后卫
    'LCB': [(1, 1, 1.0)],     # 左中卫
    'MCB': [(1, 2, 1.0)],     # 中卫
    'RCB': [(1, 3, 1.0)],     # 右中卫
    'RB': [(1, 4, 1.0)],      # 右后卫
    
    # 翼卫 - 介于后卫和防守中场之间（攻守兼备）
    'LWB': [(1, 0, 0.5), (2, 0, 0.5)],  # 左翼卫：50%后卫 + 50%防守中场
    'RWB': [(1, 4, 0.5), (2, 4, 0.5)],  # 右翼卫：50%后卫 + 50%防守中场
    
    # 防守型中场 - 第2层
    'DM': [(2, 2, 1.0)],      # 防守中场（中路）
    
    # 中场 - 第3层
    'LM': [(3, 0, 1.0)],      # 左中场
    'CM': [(3, 2, 1.0)],      # 中场
    'RM': [(3, 4, 1.0)],      # 右中场
    
    # 攻击型中场 - 介于中场和前锋之间
    'AM': [(3, 2, 0.6), (4, 2, 0.4)],  # 攻击中场：60%中场 + 40%前锋
    
    # 前锋 - 第4层（最前线，纯进攻）
    'LW': [(4, 0, 1.0)],      # 左边锋（如迪玛利亚、姆巴佩）
    'CF': [(4, 2, 1.0)],      # 中锋（如吉鲁、阿尔瓦雷斯）
    'RW': [(4, 4, 1.0)],      # 右边锋（如梅西、登贝莱）
}

print("\n📋 5×5位置矩阵设计:")
print("  纵向层级（从后到前）:")
print("    0: GK  - 守门员")
print("    1: D   - 后卫线")
print("    2: DM  - 防守中场")
print("    3: M   - 中场/攻击中场")
print("    4: F   - 前锋线")
print("  横向位置（从左到右）:")
print("    0: L   - 左")
print("    1: LC  - 左中")
print("    2: C   - 中")
print("    3: RC  - 右中")
print("    4: R   - 右")

print("\n📝 关键位置区分:")
print("  边锋 vs 翼卫:")
print("    * LW/RW (左/右边锋): 纯进攻，在第4层（前锋线）")
print("      例如: 梅西(RW), 迪玛利亚(LW), 姆巴佩(LW), 登贝莱(RW)")
print("    * LWB/RWB (左/右翼卫): 攻守兼备，跨第1层（后卫）和第2层（防守中场）")
print("      权重分配: 50%后卫 + 50%防守中场")
print("  攻击中场:")
print("    * AM: 介于中场和前锋之间")
print("      权重分配: 60%中场 + 40%前锋")

def encode_roster_position_matrix(position):
    """
    将Rosters位置编码为5×5矩阵（展平为25维向量）
    
    参数:
        position: str, Rosters位置标签（如'RW', 'LWB', 'CM'等）
    
    返回:
        vec: numpy array of shape (25,), 5×5矩阵展平后的向量
    
    示例:
        'RW' -> 矩阵[4,4]=1.0 -> 展平后第24个元素为1.0
        'LWB' -> 矩阵[1,0]=0.5, 矩阵[2,0]=0.5 -> 展平后第5和第10个元素为0.5
    """
    matrix = np.zeros((5, 5))  # 5×5矩阵
    
    if pd.notna(position) and position in POSITION_MATRIX_MAPPING:
        # 根据映射表填充矩阵
        for row, col, weight in POSITION_MATRIX_MAPPING[position]:
            matrix[row, col] = weight
    
    # 展平为25维向量（按行展平）
    vec = matrix.flatten()
    return vec

# 垂直层级和水平位置映射（保留用于Shape Graph特征）
VERTICAL_LEVELS = ['B', 'DM', 'M', 'AM', 'F']
HORIZONTAL_POSITIONS = ['L', 'LC', 'C', 'RC', 'R']

vertical_to_idx = {level: idx for idx, level in enumerate(VERTICAL_LEVELS)}
horizontal_to_idx = {pos: idx for idx, pos in enumerate(HORIZONTAL_POSITIONS)}

def encode_vertical_level_onehot(level):
    """编码垂直层级为one-hot"""
    vec = np.zeros(len(VERTICAL_LEVELS))
    if level in vertical_to_idx:
        vec[vertical_to_idx[level]] = 1
    return vec

def encode_horizontal_position_onehot(position):
    """编码水平位置为one-hot"""
    vec = np.zeros(len(HORIZONTAL_POSITIONS))
    if position in horizontal_to_idx:
        vec[horizontal_to_idx[position]] = 1
    return vec

# 定义特征类型
FEATURE_TYPES = {
    'node_features': [
        'x', 'y',  # 位置坐标
        'vx', 'vy',  # 速度分量
        'distance_to_ball',  # 到球的距离
        'roster_position_matrix',  # Rosters位置矩阵（25维）
        'vertical_level_encoded',  # 垂直层级（5维）
        'horizontal_position_encoded',  # 水平位置（5维）
    ],
    'global_features': [
        'tactical_phase_encoded',  # 战术阶段（one-hot）
        'fine_intent_encoded',  # 细粒度意图（one-hot）
        'score_diff',  # 比分差
        'time_elapsed',  # 已经过时间（归一化）
        'fifa_ranking_diff',  # FIFA排名差
        'odds_win_home',  # 主队获胜赔率
        'odds_draw',  # 平局赔率
        'odds_win_away',  # 客队获胜赔率
    ]
}

print("\n节点特征（球员特征）:")
for i, feat in enumerate(FEATURE_TYPES['node_features'], 1):
    print(f"  {i}. {feat}")

print("\n全局特征（情境特征）:")
for i, feat in enumerate(FEATURE_TYPES['global_features'], 1):
    print(f"  {i}. {feat}")

print(f"\n总节点特征维度:")
print(f"  - 基础特征: 5维 (x, y, vx, vy, distance_to_ball)")
print(f"  - Rosters位置矩阵: 25维 (5×5)")
print(f"  - 垂直层级: 5维 (B/DM/M/AM/F)")
print(f"  - 水平位置: 5维 (L/LC/C/RC/R)")
print(f"  - 总计: 5 + 25 + 5 + 5 = 40维")

print(f"\n总全局特征维度:")
print(f"  - 战术阶段和意图需根据实际类别数确定")
print(f"  - 数值特征: 5维 (score_diff, time_elapsed, fifa_ranking_diff,")
print(f"                    odds_win_home, odds_draw, odds_win_away)")

print("\n✅ 5×5位置矩阵编码优势:")
print("  1. 空间连续性: 相邻位置在矩阵中也相邻，保留空间关系")
print("  2. 位置相似性: RW和RWB都在右侧，但层级不同（前锋vs后卫+中场）")
print("  3. 多重属性: 翼卫(LWB/RWB)和攻击中场(AM)可以有多个权重")
print("  4. 语义丰富: 5×5矩阵直观反映球员在场上的位置分布")
print("  5. 模型友好: GNN可以学习位置之间的空间关系")

# 保存特征配置
feature_config = {
    'roster_positions': ALL_ROSTER_POSITIONS,
    'n_positions': len(ALL_ROSTER_POSITIONS),
    'position_matrix_mapping': POSITION_MATRIX_MAPPING,
    'feature_types': FEATURE_TYPES
}

with open(OUTPUT_DIR / 'feature_config.pkl', 'wb') as f:
    pickle.dump(feature_config, f)
print(f"\n✅ 特征配置已保存: {OUTPUT_DIR / 'feature_config.pkl'}")

=== 特征工程准备 ===

Rosters位置体系（共16个位置）:
  守门员(1): GK
  后卫(5): LB, LCB, MCB, RCB, RB
  翼卫(2): LWB, RWB
  中场(5): LM, DM, CM, AM, RM
  前锋(3): LW, CF, RW

📋 5×5位置矩阵设计:
  纵向层级（从后到前）:
    0: GK  - 守门员
    1: D   - 后卫线
    2: DM  - 防守中场
    3: M   - 中场/攻击中场
    4: F   - 前锋线
  横向位置（从左到右）:
    0: L   - 左
    1: LC  - 左中
    2: C   - 中
    3: RC  - 右中
    4: R   - 右

📝 关键位置区分:
  边锋 vs 翼卫:
    * LW/RW (左/右边锋): 纯进攻，在第4层（前锋线）
      例如: 梅西(RW), 迪玛利亚(LW), 姆巴佩(LW), 登贝莱(RW)
    * LWB/RWB (左/右翼卫): 攻守兼备，跨第1层（后卫）和第2层（防守中场）
      权重分配: 50%后卫 + 50%防守中场
  攻击中场:
    * AM: 介于中场和前锋之间
      权重分配: 60%中场 + 40%前锋

节点特征（球员特征）:
  1. x
  2. y
  3. vx
  4. vy
  5. distance_to_ball
  6. roster_position_matrix
  7. vertical_level_encoded
  8. horizontal_position_encoded

全局特征（情境特征）:
  1. tactical_phase_encoded
  2. fine_intent_encoded
  3. score_diff
  4. time_elapsed
  5. fifa_ranking_diff
  6. odds_win_home
  7. odds_draw
  8. odds_win_away

总节点特征维度:
  - 基础特征: 5维 (x, y, vx, vy, distance_to_ball)
  - Rosters位置矩阵: 25维 (5×5)

## 7. 数据加载总结

In [12]:
print("\n" + "="*60)
print("数据加载总结")
print("="*60)

data_status = {
    '阵型库': all_formations is not None,
    '追踪数据': 'tracking_data' in locals(),
    'Shape Graphs': len(graph_files) > 0,
    'Rosters数据': roster_df is not None,
    'FIFA排名': fifa_df is not None,
    '博彩赔率': odds_df is not None,
}

for name, status in data_status.items():
    symbol = "✅" if status else "❌"
    print(f"{symbol} {name}")

if all(data_status.values()):
    print("\n✅ 所有数据加载成功！")
else:
    print("\n⚠️ 部分数据缺失，请检查上述步骤")


数据加载总结
✅ 阵型库
✅ 追踪数据
✅ Shape Graphs
✅ Rosters数据
✅ FIFA排名
✅ 博彩赔率

✅ 所有数据加载成功！


## 总结

本notebook完成了以下工作：

1. ✅ 从EFPI结果中提取完整的阵型库（自动包含所有EFPI使用的阵型）
2. ✅ 加载追踪数据和战术标签
3. ✅ 加载Shape Graph序列
4. ✅ 加载Rosters数据（球员位置信息）
5. ✅ 加载FIFA排名
6. ✅ 加载博彩赔率
7. ✅ 创建球员ID到位置的映射
8. ✅ 定义特征工程框架（使用Rosters位置，移除身价薪资）

**下一步**: 运行3.2.2进行图数据集构建